# 05 — H3 Aggregation

**SDI Tehran Pipeline · Notebook 5 of 5**

## Purpose
Aggregate segment-level SDI scores to H3 hexagons (resolution 8, ~0.74 km² per hex) and produce two dashboard-ready output files uploaded to GCS.

## Methodology note — centroid assignment
Each segment is assigned to the hex containing its centroid. Long segments crossing hex boundaries are therefore attributed to a single hex. This is a known approximation; it is acceptable at H3 resolution 8 (~870m average edge length) given typical Tehran street segment lengths, but is documented as a limitation.

## CRS handling
`segments.geojson` is in UTM Zone 39N (EPSG:32639). H3 requires WGS84 (EPSG:4326). Reprojection is applied before centroid extraction.

## Inputs
- `disparity_index.csv` — segment SDI scores
- `segments.geojson` — segment geometries

## Outputs
- `hex.geojson` → `gs://sdi-tehran-assets/data/outputs/hex.geojson`
- `classification.json` → `gs://sdi-tehran-assets/data/outputs/classification.json`

## 0 · Config & imports

In [1]:
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon

sys.path.insert(0, '..')
from config import (
    DISPARITY_INDEX_CSV, SEGMENTS_GEOJSON,
    HEX_GEOJSON, CLASSIFICATION_JSON,
    GCS_BUCKET, GCS_OUTPUT_PATH, GCS_PROJECT,
    H3_RESOLUTION, MIN_SEGMENTS_PER_HEX, N_SDI_CLASSES,
    CRS_PROJECTED, CRS_GEOGRAPHIC,
    TARGET_ATTRS, SEG_ID_COL, SEGMENTS_PCA_CLUSTERED_CSV
)

SCORE_COLS = [f"score_{attr}" for attr in TARGET_ATTRS]
# SEG_ID_COL = "segment_id"
print("Config loaded.")

Config loaded.


## 1 · Load and join data

In [2]:
sdi_df  = pd.read_csv(DISPARITY_INDEX_CSV)
gdf     = gpd.read_file(SEGMENTS_GEOJSON)

print(f"SDI segments  : {len(sdi_df):,}")
print(f"Network segs  : {len(gdf):,}  |  CRS: {gdf.crs}")

gdf = gdf.merge(sdi_df, on=SEG_ID_COL, how="inner")
print(f"After join    : {len(gdf):,} segments with SDI scores")

cluster_df = pd.read_csv(SEGMENTS_PCA_CLUSTERED_CSV)[[SEG_ID_COL, "whole_CH_2"]]
gdf = gdf.merge(cluster_df, on=SEG_ID_COL, how="left")
print(f"After cluster join: whole_CH_2 null count: {gdf['whole_CH_2'].isnull().sum():,}")

SDI segments  : 279,793
Network segs  : 427,100  |  CRS: EPSG:4326
After join    : 279,793 segments with SDI scores
After cluster join: whole_CH_2 null count: 0


## 2 · Reproject to WGS84 and extract centroids

H3 functions operate in geographic coordinates (lat/lon). The source geometry is projected (UTM); we reproject before extracting centroids.

In [3]:
gdf = gdf.to_crs(CRS_GEOGRAPHIC)
gdf["centroid"] = gdf.geometry.centroid
gdf["lat"] = gdf["centroid"].y
gdf["lng"] = gdf["centroid"].x
print(f"Reprojected to {CRS_GEOGRAPHIC}")
print(f"Lat range: {gdf['lat'].min():.4f} – {gdf['lat'].max():.4f}")
print(f"Lng range: {gdf['lng'].min():.4f} – {gdf['lng'].max():.4f}")

C:\Users\sepehr\AppData\Local\Temp\ipykernel_2320\1285871745.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid


Reprojected to EPSG:4326
Lat range: 34.3552 – 36.3521
Lng range: 49.8551 – 53.2837


## 3 · Assign H3 hex IDs

In [4]:
gdf["hex_id"] = gdf.apply(
    lambda row: h3.latlng_to_cell(row["lat"], row["lng"], H3_RESOLUTION), axis=1
)
n_hexes = gdf["hex_id"].nunique()
print(f"H3 resolution {H3_RESOLUTION}: {n_hexes:,} unique hexes")

H3 resolution 8: 13,848 unique hexes


## 4 · Aggregate to hexes

In [5]:
agg_dict = {"sdi": "mean", "n_segments": ("sdi", "count")}
for col in SCORE_COLS:
    agg_dict[col] = "mean"

hex_agg = gdf.groupby("hex_id").agg(
    sdi         = ("sdi",        "mean"),
    n_segments  = ("sdi",        "count"),
    whole_CH_2  = ("whole_CH_2", lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan),
    **{col: (col, "mean") for col in SCORE_COLS}
).reset_index()

# Filter empty hexes
hex_agg = hex_agg[hex_agg["n_segments"] >= MIN_SEGMENTS_PER_HEX].copy()
print(f"Hexes after filtering (n_segments >= {MIN_SEGMENTS_PER_HEX}): {len(hex_agg):,}")

Hexes after filtering (n_segments >= 1): 13,848


## 5 · Compute quantile classification breaks

In [6]:
classification = {}
classify_cols = ["sdi"] + SCORE_COLS

for col in classify_cols:
    quantiles = np.linspace(0, 1, N_SDI_CLASSES + 1)
    breaks = np.quantile(hex_agg[col].dropna(), quantiles).tolist()
    classification[col] = {"breaks": breaks, "n_classes": N_SDI_CLASSES}

    # Assign class label (1-indexed)
    unique_breaks = pd.unique(breaks)
    unique_breaks.sort()
    hex_agg[f"{col}_class"] = pd.cut(
        hex_agg[col],
        bins=unique_breaks,
        labels=range(1, len(unique_breaks)),
        include_lowest=True
    ).astype("Int64")

print("Classification breaks computed.")
for col, info in classification.items():
    print(f"  {col}: {[round(b,3) for b in info['breaks']]}")

Classification breaks computed.
  sdi: [0.475, 0.941, 0.997, 0.997, 0.997, 0.999]
  score_policy_era: [0.2, 0.914, 1.0, 1.0, 1.0, 1.0]
  score_dilap_index: [0.667, 1.0, 1.0, 1.0, 1.0, 1.0]
  score_landuse: [0.25, 0.25, 0.25, 0.25, 0.25, 0.25]
  score_mean_area: [0.125, 0.523, 0.544, 0.555, 0.563, 0.884]
  score_perimeter: [0.254, 0.625, 0.649, 0.664, 0.675, 0.939]
  score_mean_pop: [0.033, 0.185, 0.218, 0.243, 0.259, 0.486]


C:\Users\sepehr\AppData\Local\Temp\ipykernel_2320\4122018568.py:10: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_breaks = pd.unique(breaks)
C:\Users\sepehr\AppData\Local\Temp\ipykernel_2320\4122018568.py:10: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_breaks = pd.unique(breaks)
C:\Users\sepehr\AppData\Local\Temp\ipykernel_2320\4122018568.py:10: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_breaks = pd.unique(breaks)
C:\Users\sepehr\AppData\Local\Temp\ipykernel_2320\4122018568.py:10: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_breaks = pd.uni

## 6 · Build hex polygons and GeoDataFrame

In [7]:
def hex_to_polygon(hex_id):
    boundary = h3.cell_to_boundary(hex_id)
    # h3 returns (lat, lng) tuples; Shapely needs (lng, lat)
    return Polygon([(lng, lat) for lat, lng in boundary])

hex_agg["geometry"] = hex_agg["hex_id"].apply(hex_to_polygon)
hex_gdf = gpd.GeoDataFrame(hex_agg, geometry="geometry", crs=CRS_GEOGRAPHIC)

print(f"Hex GeoDataFrame: {len(hex_gdf):,} rows  |  CRS: {hex_gdf.crs}")

Hex GeoDataFrame: 13,848 rows  |  CRS: EPSG:4326


## 7 · Save outputs

In [8]:
hex_gdf.to_file(HEX_GEOJSON, driver="GeoJSON")
print(f"Saved: {HEX_GEOJSON}")

with open(CLASSIFICATION_JSON, "w") as f:
    json.dump(classification, f, indent=2)
print(f"Saved: {CLASSIFICATION_JSON}")

Saved: D:\SID\data\outputs\hex.geojson
Saved: D:\SID\data\outputs\classification.json


## 8 · Upload to GCS

Requires `google-cloud-storage` and authenticated credentials (`gcloud auth application-default login` or a service account key).

In [9]:
from google.cloud import storage

client = storage.Client(project=GCS_PROJECT)
bucket = client.bucket(GCS_BUCKET)

for local_path, gcs_name in [
    (HEX_GEOJSON,       "hex.geojson"),
    (CLASSIFICATION_JSON, "classification.json"),
]:
    blob = bucket.blob(f"{GCS_OUTPUT_PATH}/{gcs_name}")
    blob.upload_from_filename(local_path)
    print(f"Uploaded: gs://{GCS_BUCKET}/{GCS_OUTPUT_PATH}/{gcs_name}")

print("\nAll outputs uploaded to GCS.")

C:\Users\sepehr\AppData\Local\Programs\Python\Python39\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\sepehr\AppData\Local\Programs\Python\Python39\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\sepehr\AppData\Local\Programs\Python\Python39\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supporte

Uploaded: gs://sdi-tehran-assets/data/outputs/hex.geojson
Uploaded: gs://sdi-tehran-assets/data/outputs/classification.json

All outputs uploaded to GCS.
